In [ ]:
!pip -q install -U ultralytics roboflow

In [ ]:
from ultralytics import YOLO
from roboflow import Roboflow
from pathlib import Path
import torch
import os

print(torch.cuda.get_device_name(0))

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="8MqIaual7OyR3dQXZMRj")
project = rf.workspace("apaar-mathur").project("safety-food-system-aecc8-znomq")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
import yaml

yaml_file = "/content/Safety-Food-System-2/data.yaml"

with open(yaml_file) as f:
    cfg = yaml.safe_load(f)

print(cfg["names"])

In [ ]:
from pathlib import Path
import shutil
import yaml

NEW_DATASET = Path("/content/hairnet_finetune")

if NEW_DATASET.exists():
    shutil.rmtree(NEW_DATASET)

for split in ["train","valid","test"]:

    (NEW_DATASET/split/"images").mkdir(parents=True, exist_ok=True)
    (NEW_DATASET/split/"labels").mkdir(parents=True, exist_ok=True)

    img_dir = DATASET/split/"images"
    lbl_dir = DATASET/split/"labels"

    for img in img_dir.glob("*"):

        label = lbl_dir/f"{img.stem}.txt"

        if not label.exists():
            continue

        new_lines = []

        with open(label) as f:

            for line in f:

                cls,*box = line.strip().split()

                cls = int(cls)

                if cls == 1:
                    new_lines.append(
                        "4 " + " ".join(box)
                    )

        if len(new_lines)==0:
            continue

        shutil.copy2(img, NEW_DATASET/split/"images"/img.name)

        with open(
            NEW_DATASET/split/"labels"/label.name,
            "w"
        ) as f:
            f.write("\n".join(new_lines))

In [ ]:
import yaml

classes = [
    "apron",
    "no_apron",
    "gloves",
    "no_gloves",
    "hairnet",
    "no_hairnet",
    "with_mask",
    "without_mask",
]

cfg = {

    "path": str(NEW_DATASET),

    "train":"train/images",

    "val":"valid/images",

    "test":"test/images",

    "nc":8,

    "names":classes
}

with open(
    NEW_DATASET/"data.yaml",
    "w"
) as f:

    yaml.dump(cfg,f,sort_keys=False)

print("Done")

In [ ]:
from collections import Counter

counter = Counter()

for txt in NEW_DATASET.rglob("*.txt"):

    with open(txt) as f:

        for line in f:

            counter[int(line.split()[0])] += 1

print(counter)

In [ ]:
model = YOLO("best.pt")

In [ ]:
from ultralytics import YOLO

model = YOLO("best.pt")

model.train(

    data=str(NEW_DATASET/"data.yaml"),

    epochs=15,

    freeze=10,

    pretrained=False,

    imgsz=(640, 1024),

    batch=16,

    rect=True,

    cache=True,

    optimizer="AdamW",

    lr0=2e-4,

    lrf=1e-2,

    cos_lr=True,

    weight_decay=5e-4,

    warmup_epochs=2,

    patience=8,

    workers=2,

    amp=True,

    device=0,

    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,

    translate=0.03,
    scale=0.20,

    fliplr=0.5,

    mosaic=0.30,

    mixup=0,

    close_mosaic=5,

    project="HairnetFineTune",

    name="stage1"
)

In [ ]:
model = YOLO(
    "/content/HairnetFineTune/stage1/weights/best.pt"
)

model.train(

    data=str(NEW_DATASET/"data.yaml"),

    epochs=20,

    freeze=0,

    pretrained=False,

    imgsz=(640, 1024),

    batch=16,

    rect=True,

    cache=True,

    optimizer="AdamW",

    lr0=1e-4,

    lrf=1e-2,

    cos_lr=True,

    patience=10,

    workers=2,

    amp=True,

    device=0,

    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.2,

    translate=0.02,
    scale=0.15,

    fliplr=0.5,

    mosaic=0.15,

    close_mosaic=5,

    project="HairnetFineTune",

    name="stage2"
)

In [ ]:
best = YOLO(
    "/content/KitchenCam/hygiene_finetuned/weights/best.pt"
)

metrics = best.val(
    data=str(DATASET / "data.yaml"),
    split="test",
    plots=True
)

print(metrics)

In [ ]:
best = YOLO(
    "/content/HairnetFineTune/stage2/weights/best.pt"
)

best.val(
    data=str(NEW_DATASET/"data.yaml"),
    split="test",
    plots=True
)